# 🛸 Monarch Industries — NEO Risk Scorer
## Step 3: XGBoost Regressor + SHAP Analysis

**Goal:** Train a model to predict continuous risk scores from orbital elements alone.

**Key design:** Model inputs (X) are orbital features only. Target (y) is the physics-derived risk score — NOT NASA's binary hazard flag.

---
**Upload your `neo_data.db` file using the cell below before running anything.**

In [ ]:
# ── Cell 1: Upload neo_data.db from your local machine ──────────────────────
from google.colab import files
print('Upload your neo_data.db file...')
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
!pip install xgboost shap matplotlib seaborn scikit-learn --quiet
print('All dependencies installed.')

In [ ]:
# ── Cell 3: Imports ───────────────────────────────────────────────────────────
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings
warnings.filterwarnings('ignore')

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

print('Imports OK')

In [ ]:
# ── Cell 4: Load data from SQLite ─────────────────────────────────────────────
conn = sqlite3.connect('neo_data.db')

df = pd.read_sql_query("""
    SELECT
        f.neo_id,
        n.name,

        -- ── Features (X) — orbital elements only ──────────────────────────
        f.absolute_magnitude_h,
        f.diameter_mean_km,
        f.eccentricity,
        f.semi_major_axis,
        f.inclination,
        f.perihelion_distance,
        f.aphelion_distance,
        f.orbital_period,
        f.perihelion_crosses_earth_orbit,
        f.ecc_x_inclination,
        f.orbit_class_encoded,

        -- ── Target (y) ────────────────────────────────────────────────────
        f.risk_score_normalized,

        -- ── Reference only (NOT passed to model) ─────────────────────────
        f.risk_tier,
        f.ca_min_miss_au,
        f.ca_count,
        n.is_potentially_hazardous

    FROM neo_features f
    JOIN neo_objects n ON f.neo_id = n.id
    WHERE f.risk_score_normalized IS NOT NULL
""", conn)

conn.close()

print(f'Loaded {len(df):,} NEOs')
print(f'Columns: {list(df.columns)}')
df.head(3)

In [ ]:
# ── Cell 5: EDA — Distribution check ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Risk Score Distribution', fontsize=14, fontweight='bold')

# Raw risk score (log scale)
axes[0].hist(df['risk_score_normalized'], bins=100, color='#e63946', edgecolor='none')
axes[0].set_title('Z-Score Distribution')
axes[0].set_xlabel('risk_score_normalized (z)')
axes[0].set_ylabel('Count')

# Tier breakdown
tier_order = ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW']
tier_colors = ['#e63946', '#f4a261', '#2a9d8f', '#457b9d']
tier_counts = df['risk_tier'].value_counts().reindex(tier_order)
axes[1].bar(tier_counts.index, tier_counts.values, color=tier_colors)
axes[1].set_title('Risk Tier Distribution')
axes[1].set_ylabel('Count')

# Risk score vs eccentricity
sample = df.sample(min(5000, len(df)), random_state=42)
axes[2].scatter(sample['eccentricity'], sample['risk_score_normalized'],
                alpha=0.3, s=5, c='#e63946')
axes[2].set_title('Risk Score vs Eccentricity')
axes[2].set_xlabel('Eccentricity')
axes[2].set_ylabel('Risk Score (z)')

plt.tight_layout()
plt.savefig('risk_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Distribution looks healthy if z-scores are roughly bell-curved.')

In [ ]:
# ── Cell 6: Feature matrix + train/test split ─────────────────────────────────
FEATURE_COLS = [
    'absolute_magnitude_h',
    'diameter_mean_km',
    'eccentricity',
    'semi_major_axis',
    'inclination',
    'perihelion_distance',
    'aphelion_distance',
    'orbital_period',
    'perihelion_crosses_earth_orbit',
    'ecc_x_inclination',
    'orbit_class_encoded',
]
TARGET_COL = 'risk_score_normalized'

# Drop rows with any missing features
df_clean = df[FEATURE_COLS + [TARGET_COL, 'name', 'risk_tier']].dropna()
print(f'Clean dataset: {len(df_clean):,} rows (dropped {len(df) - len(df_clean):,} with nulls)')

X = df_clean[FEATURE_COLS].values
y = df_clean[TARGET_COL].values

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df_clean.index, test_size=0.2, random_state=42
)

print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

In [ ]:
# ── Cell 7: Train XGBoost Regressor ──────────────────────────────────────────
model = XGBRegressor(
    n_estimators      = 500,
    max_depth         = 6,
    learning_rate     = 0.05,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_weight  = 3,
    reg_alpha         = 0.1,    # L1 regularization
    reg_lambda        = 1.0,    # L2 regularization
    objective         = 'reg:squarederror',
    eval_metric       = 'rmse',
    early_stopping_rounds = 30,
    random_state      = 42,
    n_jobs            = -1,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=50
)

print(f'\nBest iteration: {model.best_iteration}')

In [ ]:
# ── Cell 8: Evaluation ────────────────────────────────────────────────────────
y_pred_train = model.predict(X_train)
y_pred_test  = model.predict(X_test)

train_r2   = r2_score(y_train, y_pred_train)
test_r2    = r2_score(y_test,  y_pred_test)
test_rmse  = np.sqrt(mean_squared_error(y_test, y_pred_test))
test_mae   = mean_absolute_error(y_test, y_pred_test)

print('━' * 45)
print('  MODEL EVALUATION')
print('━' * 45)
print(f'  Train R²   : {train_r2:.4f}')
print(f'  Test  R²   : {test_r2:.4f}   ← key metric')
print(f'  Test  RMSE : {test_rmse:.4f}')
print(f'  Test  MAE  : {test_mae:.4f}')
print('━' * 45)

if test_r2 > 0.85:
    print('  ✅ Strong — orbital elements predict risk well')
elif test_r2 > 0.65:
    print('  ⚠️  Decent — some risk variance not captured by orbital elements alone')
else:
    print('  ❌ Weak — check for data issues or feature leakage')

# Overfitting check
gap = train_r2 - test_r2
print(f'\n  Train/Test R² gap: {gap:.4f}', '✅ OK' if gap < 0.1 else '⚠️ Possible overfit')

In [ ]:
# ── Cell 9: Learning curve (loss over boosting rounds) ───────────────────────
results = model.evals_result()
train_loss = results['validation_0']['rmse']
test_loss  = results['validation_1']['rmse']

plt.figure(figsize=(10, 4))
plt.plot(train_loss, label='Train RMSE', color='#457b9d')
plt.plot(test_loss,  label='Test RMSE',  color='#e63946')
plt.axvline(model.best_iteration, color='gray', linestyle='--', label=f'Best iter ({model.best_iteration})')
plt.xlabel('Boosting Round')
plt.ylabel('RMSE')
plt.title('XGBoost Learning Curve — NEO Risk Score')
plt.legend()
plt.tight_layout()
plt.savefig('learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 10: Predicted vs Actual scatter ─────────────────────────────────────
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_test, alpha=0.3, s=8, c='#e63946')
lims = [min(y_test.min(), y_pred_test.min()),
        max(y_test.max(), y_pred_test.max())]
plt.plot(lims, lims, 'k--', linewidth=1, label='Perfect prediction')
plt.xlabel('Actual Risk Score (z)')
plt.ylabel('Predicted Risk Score (z)')
plt.title(f'Predicted vs Actual  |  Test R² = {test_r2:.4f}')
plt.legend()
plt.tight_layout()
plt.savefig('predicted_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 11: SHAP Analysis ────────────────────────────────────────────────────
# SHAP tells us WHICH orbital features drive risk predictions
# and by HOW MUCH — this is the scientifically interesting output

print('Computing SHAP values (may take ~1-2 min)...')
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Summary bar plot — global feature importance
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values, X_test,
    feature_names=FEATURE_COLS,
    plot_type='bar',
    show=False
)
plt.title('SHAP Feature Importance — Which orbital elements predict risk?')
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 12: SHAP Beeswarm — direction + magnitude ───────────────────────────
# Shows not just importance but HOW each feature pushes the score up or down

plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values, X_test,
    feature_names=FEATURE_COLS,
    show=False
)
plt.title('SHAP Beeswarm — How orbital elements drive risk score')
plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

print()
print('Reading the beeswarm:')
print('  Red dots  = high feature value')
print('  Blue dots = low feature value')
print('  Right of 0 = pushes risk UP')
print('  Left of 0  = pushes risk DOWN')

In [ ]:
# ── Cell 13: Final risk ranking table ────────────────────────────────────────
df_test = df_clean.iloc[idx_test].copy()
df_test['predicted_risk'] = y_pred_test
df_test['actual_risk']    = y_test
df_test['error']          = abs(df_test['predicted_risk'] - df_test['actual_risk'])

top20 = df_test.sort_values('predicted_risk', ascending=False).head(20)

print('\n TOP 20 RISKIEST NEOs (model-predicted ranking)')
print('─' * 72)
print(f'{"Rank":<5} {"Name":<32} {"Predicted z":>11} {"Actual z":>9} {"Tier":<10}')
print('─' * 72)
for i, (_, row) in enumerate(top20.iterrows(), 1):
    name = str(row['name'])[:30]
    print(f'{i:<5} {name:<32} {row["predicted_risk"]:>11.4f} {row["actual_risk"]:>9.4f} {row["risk_tier"]:<10}')
print('─' * 72)

In [ ]:
# ── Cell 14: Save model + download all outputs ────────────────────────────────
import pickle, zipfile, os

# Save model
model.save_model('neo_risk_xgboost.json')
with open('feature_cols.pkl', 'wb') as f:
    pickle.dump(FEATURE_COLS, f)

# Zip everything
output_files = [
    'neo_risk_xgboost.json',
    'feature_cols.pkl',
    'risk_distribution.png',
    'learning_curve.png',
    'predicted_vs_actual.png',
    'shap_importance.png',
    'shap_beeswarm.png',
]

with zipfile.ZipFile('monarch_neo_outputs.zip', 'w') as z:
    for f in output_files:
        if os.path.exists(f):
            z.write(f)
            print(f'  Added: {f}')

print('\nDownloading outputs...')
files.download('monarch_neo_outputs.zip')
print('Done. Check your downloads folder.')